[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-format-management/tabular.ipynb)

# Tabular data

Analysis tools want tables: one row per observation, one column per variable.
This notebook reads and writes CSV with plain Python, the `csv` module, and pandas, then covers separators, data types, compression, Parquet, files too large for memory, and the conversion between JSON and tables.

Sections: Plain Python, The csv module, pandas, Broken CSV, Other separators, Data types, Compression, Parquet, Large files, Tabular to JSON, JSON to tabular.

Packages beyond the standard library: `pandas`, `pyarrow`.
Install them with `uv add pandas pyarrow` (or `pip install`).

In [1]:
# The sample files live next to this notebook in data/. In Colab they do not
# exist yet, so this cell downloads them from the course repository.
import pathlib
import urllib.request

base_url = "https://raw.githubusercontent.com/YangKCLab/social-media-analysis/main/docs/topics/data-format-management/data/"
pathlib.Path("data").mkdir(exist_ok=True)
for name in ["sample.csv", "sample_broken.csv", "fips.csv", "sample.json"]:
    path = pathlib.Path("data") / name
    if not path.exists():
        urllib.request.urlretrieve(base_url + name, path)
print(sorted(p.name for p in pathlib.Path("data").iterdir()))

['fips.csv', 'sample.csv', 'sample.json', 'sample_broken.csv', 'sample_broken.json']


## Plain Python

CSV stands for comma-separated values. The first line usually names the columns; every other line is one record, with a comma between values.
It is plain text, so a text editor, a spreadsheet, and `head` all open it.

In [2]:
print(open("data/sample.csv").read())

name,age,city,occupation,salary
Alice Johnson,28,New York,Software Engineer,85000
Bob Smith,34,San Francisco,Data Scientist,95000
Carol Davis,29,Boston,Product Manager,75000
David Wilson,31,Seattle,UX Designer,70000
Eva Brown,26,Austin,Marketing Specialist,60000


The obvious way to read it is to split each line on commas.

In [3]:
csv_content = []
with open("data/sample.csv") as f:
    for line in f:
        csv_content.append(line.strip().split(","))

csv_content

[['name', 'age', 'city', 'occupation', 'salary'],
 ['Alice Johnson', '28', 'New York', 'Software Engineer', '85000'],
 ['Bob Smith', '34', 'San Francisco', 'Data Scientist', '95000'],
 ['Carol Davis', '29', 'Boston', 'Product Manager', '75000'],
 ['David Wilson', '31', 'Seattle', 'UX Designer', '70000'],
 ['Eva Brown', '26', 'Austin', 'Marketing Specialist', '60000']]

Two things are wrong with this. Every value is a string, so `'28'` is not a number yet.
And a comma inside a value, such as a name written `Davis, Carol`, splits into two fields.
Real CSV handles that by wrapping the value in double quotes, and `split(",")` does not know about the quotes.

## The csv module

The standard library's `csv` module reads and writes the quoting rules correctly.
`csv.writer` quotes a value when it needs to.

In [4]:
import csv

with open("data/quoted.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age", "city"])
    writer.writerow(["Davis, Carol", 29, "Boston"])
    writer.writerow(['Bob "Bobby" Smith', 34, "San Francisco"])

print(open("data/quoted.csv").read())

name,age,city
"Davis, Carol",29,Boston
"Bob ""Bobby"" Smith",34,San Francisco



`csv.reader` returns one list per row, and `csv.DictReader` returns one dictionary per row, keyed by the header line.
Both split `"Davis, Carol"` as one value.

In [5]:
with open("data/quoted.csv", newline="") as f:
    for row in csv.reader(f):
        print(row)

['name', 'age', 'city']
['Davis, Carol', '29', 'Boston']
['Bob "Bobby" Smith', '34', 'San Francisco']


In [6]:
with open("data/quoted.csv", newline="") as f:
    for row in csv.DictReader(f):
        print(row["name"], "->", row["city"])

Davis, Carol -> Boston
Bob "Bobby" Smith -> San Francisco


## pandas

[pandas](https://pandas.pydata.org/) is the standard Python library for tables.
A table is a `DataFrame`; `read_csv` builds one from a file, and it guesses a type for each column.

In [7]:
import pandas as pd

csv_df = pd.read_csv("data/sample.csv")
csv_df

,name,age,city,occupation,salary
0,Alice Johnson,28,New York,Software Engineer,85000
1,Bob Smith,34,San Francisco,Data Scientist,95000
2,Carol Davis,29,Boston,Product Manager,75000
3,David Wilson,31,Seattle,UX Designer,70000
4,Eva Brown,26,Austin,Marketing Specialist,60000


In [8]:
csv_df.dtypes

name            str
age           int64
city            str
occupation      str
salary        int64
dtype: object

A column is selected by name and behaves like a typed array. `age` came back as integers, so arithmetic works on it.

In [9]:
csv_df["age"]

0    28
1    34
2    29
3    31
4    26
Name: age, dtype: int64

In [10]:
print(csv_df["age"].mean())

29.6


`to_csv` writes a table back out. Pass `index=False`, otherwise pandas writes its row numbers as an extra first column.

In [11]:
csv_df.to_csv("data/sample_copy.csv", index=False)
print(open("data/sample_copy.csv").read())

name,age,city,occupation,salary
Alice Johnson,28,New York,Software Engineer,85000
Bob Smith,34,San Francisco,Data Scientist,95000
Carol Davis,29,Boston,Product Manager,75000
David Wilson,31,Seattle,UX Designer,70000
Eva Brown,26,Austin,Marketing Specialist,60000



## Broken CSV

`data/sample_broken.csv` has two damaged rows: line 2 has six values instead of five, and line 4 has an unquoted comma inside a name.
pandas does not raise an error. It reads the file, and the result is wrong in a way that is easy to miss.

In [12]:
print(open("data/sample_broken.csv").read())

name,age,city,occupation,salary
Alice Johnson,28,New York,Software Engineer,Data Scientist,85000
Bob Smith,34,San Francisco,Data Scientist,95000
Davis, Carol,29,Boston,Product Manager,75000
David Wilson,31,Seattle,UX Designer,70000
Eva Brown,26,Austin,Marketing Specialist,60000



In [13]:
pd.read_csv("data/sample_broken.csv")

,name,age,city,occupation,salary
Alice Johnson,28,New York,Software Engineer,Data Scientist,85000.0
Bob Smith,34,San Francisco,Data Scientist,95000,NaN
Davis,Carol,29,Boston,Product Manager,75000.0
David Wilson,31,Seattle,UX Designer,70000,NaN
Eva Brown,26,Austin,Marketing Specialist,60000,NaN


Because the first data row has one value more than the header, pandas decided the first column is the row index, and every column shifted left.
When a CSV comes from someone else, look at the first rows and the `dtypes` before analyzing anything.
A column of numbers that arrives as `object` is the usual sign of a shifted or damaged row.

## Other separators

The separator does not have to be a comma. Tabs (`.tsv`) and pipes (`|`) are common when the values themselves contain commas.
pandas assumes a comma unless told otherwise.

In [14]:
csv_df.to_csv("data/sample_pipe.csv", sep="|", index=False)
print(open("data/sample_pipe.csv").read())

name|age|city|occupation|salary
Alice Johnson|28|New York|Software Engineer|85000
Bob Smith|34|San Francisco|Data Scientist|95000
Carol Davis|29|Boston|Product Manager|75000
David Wilson|31|Seattle|UX Designer|70000
Eva Brown|26|Austin|Marketing Specialist|60000



In [15]:
pd.read_csv("data/sample_pipe.csv")

,name|age|city|occupation|salary
0,Alice Johnson|28|New York|Software Engineer|85000
1,Bob Smith|34|San Francisco|Data Scientist|95000
2,Carol Davis|29|Boston|Product Manager|75000
3,David Wilson|31|Seattle|UX Designer|70000
4,Eva Brown|26|Austin|Marketing Specialist|60000


Without `sep="|"`, every line is one value in one column. With it, the table is back.

In [16]:
pd.read_csv("data/sample_pipe.csv", sep="|")

,name,age,city,occupation,salary
0,Alice Johnson,28,New York,Software Engineer,85000
1,Bob Smith,34,San Francisco,Data Scientist,95000
2,Carol Davis,29,Boston,Product Manager,75000
3,David Wilson,31,Seattle,UX Designer,70000
4,Eva Brown,26,Austin,Marketing Specialist,60000


In [17]:
csv_df.to_csv("data/sample_tab.tsv", sep="\t", index=False)
pd.read_csv("data/sample_tab.tsv", sep="\t")

,name,age,city,occupation,salary
0,Alice Johnson,28,New York,Software Engineer,85000
1,Bob Smith,34,San Francisco,Data Scientist,95000
2,Carol Davis,29,Boston,Product Manager,75000
3,David Wilson,31,Seattle,UX Designer,70000
4,Eva Brown,26,Austin,Marketing Specialist,60000


## Data types

CSV has no types. Everything is text, and the reader guesses.
`data/fips.csv` holds county FIPS codes, which are five-digit identifiers with leading zeros.
pandas sees digits and guesses integers, and the leading zero disappears.

In [18]:
print(open("data/fips.csv").read())

state,county,fips
Alabama,Autauga,01001
Alabama,Baldwin,01003
Alabama,Barbour,01005
Alabama,Bibb,01007



In [19]:
fips_df = pd.read_csv("data/fips.csv")
fips_df

,state,county,fips
0,Alabama,Autauga,1001
1,Alabama,Baldwin,1003
2,Alabama,Barbour,1005
3,Alabama,Bibb,1007


In [20]:
fips_df.dtypes

state       str
county      str
fips      int64
dtype: object

`dtype` tells `read_csv` the type of a column. Identifiers (FIPS codes, ZIP codes, user IDs, post IDs) are strings, even when they look like numbers: they are never added together, and a leading zero or a 19-digit ID does not survive as an integer.

In [21]:
fips_str_df = pd.read_csv("data/fips.csv", dtype={"fips": "str"})
fips_str_df

,state,county,fips
0,Alabama,Autauga,01001
1,Alabama,Baldwin,01003
2,Alabama,Barbour,01005
3,Alabama,Bibb,01007


In [22]:
fips_str_df.dtypes

state     str
county    str
fips      str
dtype: object

## Compression

pandas reads and writes gzip directly. The `.gz` extension is enough; `compression="gzip"` makes it explicit.

In [23]:
fips_str_df.to_csv("data/fips.csv.gz", index=False)
pd.read_csv("data/fips.csv.gz", dtype={"fips": "str"})

,state,county,fips
0,Alabama,Autauga,01001
1,Alabama,Baldwin,01003
2,Alabama,Barbour,01005
3,Alabama,Bibb,01007


## Parquet

CSV has no types and must be parsed from the first byte. [Apache Parquet](https://parquet.apache.org/) is a binary, column-oriented format: every column carries its type, the file is compressed, and a reader can load two columns out of fifty without touching the rest.
pandas reads and writes it through the `pyarrow` package.

In [24]:
fips_str_df.to_parquet("data/fips.parquet")
pd.read_parquet("data/fips.parquet")

,state,county,fips
0,Alabama,Autauga,01001
1,Alabama,Baldwin,01003
2,Alabama,Barbour,01005
3,Alabama,Bibb,01007


The string type of `fips` survived the round trip. Nobody has to remember the `dtype` argument.

In [25]:
pd.read_parquet("data/fips.parquet").dtypes

state     str
county    str
fips      str
dtype: object

File size depends on the data and on the compression codec. On this table of 200,000 rows, `csv.gz` beats Parquet's default codec (`snappy`), and Parquet with `zstd` beats both.

In [26]:
import os

big = pd.DataFrame({
    "id": range(200_000),
    "platform": ["bluesky", "4chan", "youtube"] * 66_666 + ["bluesky", "4chan"],
    "likes": [i % 50 for i in range(200_000)],
})
big.to_csv("data/big.csv", index=False)
big.to_csv("data/big.csv.gz", index=False)
big.to_parquet("data/big.parquet")
big.to_parquet("data/big_zstd.parquet", compression="zstd")

for name in ["big.csv", "big.csv.gz", "big.parquet", "big_zstd.parquet"]:
    print(f"{name:18s} {os.path.getsize('data/' + name) / 1e6:5.2f} MB")

big.csv             3.32 MB
big.csv.gz          0.53 MB
big.parquet         1.09 MB
big_zstd.parquet    0.47 MB


The advantage that does not depend on the data: a CSV reader must parse every byte of every row, and a Parquet reader loads only the columns you ask for.

In [27]:
import timeit

t_csv = timeit.timeit(lambda: pd.read_csv("data/big.csv"), number=5) / 5
t_one = timeit.timeit(lambda: pd.read_parquet("data/big.parquet", columns=["likes"]), number=5) / 5
print(f"read_csv, all columns:         {t_csv * 1000:6.1f} ms")
print(f"read_parquet, one column:      {t_one * 1000:6.1f} ms")

read_csv, all columns:           26.9 ms
read_parquet, one column:         0.9 ms


## Large files

`read_csv` loads the whole file into memory. When the file is larger than the memory you have, `chunksize` returns an iterator of DataFrames and you process one piece at a time.
The same pattern applies to a JSONL file read line by line.

In [28]:
total = 0
rows = 0
for chunk in pd.read_csv("data/big.csv", chunksize=50_000):
    total += chunk["likes"].sum()
    rows += len(chunk)

print(rows, "rows,", total, "likes")

200000 rows, 4900000 likes


## Tabular to JSON

A table converts to JSON without loss: one object per row, column names as keys.

In [29]:
print(fips_str_df.to_json(orient="records", indent=2))

[
  {
    "state":"Alabama",
    "county":"Autauga",
    "fips":"01001"
  },
  {
    "state":"Alabama",
    "county":"Baldwin",
    "fips":"01003"
  },
  {
    "state":"Alabama",
    "county":"Barbour",
    "fips":"01005"
  },
  {
    "state":"Alabama",
    "county":"Bibb",
    "fips":"01007"
  }
]


`to_dict(orient="records")` gives the same rows as Python dictionaries, which is what you want for writing JSONL.

In [30]:
import json

with open("data/fips.jsonl", "w") as f:
    for row in fips_str_df.to_dict(orient="records"):
        f.write(json.dumps(row) + "\n")

print(open("data/fips.jsonl").read())

{"state": "Alabama", "county": "Autauga", "fips": "01001"}
{"state": "Alabama", "county": "Baldwin", "fips": "01003"}
{"state": "Alabama", "county": "Barbour", "fips": "01005"}
{"state": "Alabama", "county": "Bibb", "fips": "01007"}



## JSON to tabular

The other direction is the hard one.
A JSON object nests: `address` holds an object, `hobbies` holds a list. A table cell holds one value.
`pd.json_normalize` flattens nested objects into dotted column names.

In [31]:
with open("data/sample.json") as f:
    sample_obj = json.load(f)

flat = pd.json_normalize(sample_obj)
flat.columns.tolist()

['name',
 'age',
 'isStudent',
 'height',
 'hobbies',
 'favoriteColors',
 'spouse',
 'lastLogin',
 'isActive',
 'address.street',
 'address.city',
 'address.zipCode',
 'address.coordinates.latitude',
 'address.coordinates.longitude',
 'education.degree',
 'education.field',
 'education.university',
 'education.graduationYear',
 'education.gpa',
 'socialMedia.twitter',
 'socialMedia.linkedin',
 'socialMedia.github']

In [32]:
flat[["name", "age", "address.city", "address.coordinates.latitude", "hobbies"]]

,name,age,address.city,address.coordinates.latitude,hobbies
0,Alice Johnson,28,New York,40.7128,"[reading, cycling, photography]"


The list in `hobbies` is still a list inside one cell.
There are two ways out. `explode` makes one row per list item, which is the right shape for a second table (one row per person-hobby pair).
Or keep the list as a JSON string if it is only carried along and never analyzed.

In [33]:
flat[["name", "hobbies"]].explode("hobbies")

,name,hobbies
0,Alice Johnson,reading
0,Alice Johnson,cycling
0,Alice Johnson,photography


The same steps turn API responses into an analysis table.
Three Bluesky-shaped posts, with a nested `author` and an optional `embed`:

In [34]:
posts = [
    {"uri": "at://did:plc:a/app.bsky.feed.post/1", "author": {"did": "did:plc:a", "handle": "ana.bsky.social"},
     "record": {"text": "first post", "createdAt": "2025-08-26T04:36:32Z"}, "likeCount": 3},
    {"uri": "at://did:plc:b/app.bsky.feed.post/2", "author": {"did": "did:plc:b", "handle": "bo.bsky.social"},
     "record": {"text": "with a picture", "createdAt": "2025-08-26T05:00:00Z"}, "likeCount": 10,
     "embed": {"$type": "app.bsky.embed.images#view", "images": [{"alt": "a cat"}]}},
    {"uri": "at://did:plc:a/app.bsky.feed.post/3", "author": {"did": "did:plc:a", "handle": "ana.bsky.social"},
     "record": {"text": "third post", "createdAt": "2025-08-27T09:15:00Z"}, "likeCount": 0},
]

posts_df = pd.json_normalize(posts)
posts_df.columns.tolist()

['uri',
 'likeCount',
 'author.did',
 'author.handle',
 'record.text',
 'record.createdAt',
 'embed.$type',
 'embed.images']

`json_normalize` uses the union of every field it saw, so `embed.$type` exists for every row and is `NaN` where the post had no embed.
Pick the columns the question needs, rename them, and fix the types. The result is a table you can save as CSV or Parquet and analyze.

In [35]:
table = posts_df[["uri", "author.handle", "record.createdAt", "record.text", "likeCount"]].rename(columns={
    "author.handle": "handle",
    "record.createdAt": "created_at",
    "record.text": "text",
    "likeCount": "likes",
})
table["created_at"] = pd.to_datetime(table["created_at"])
table

,uri,handle,created_at,text,likes
0,at://did:plc:a/app.bsky.feed.post/1,ana.bsky.social,2025-08-26 04:36:32+00:00,first post,3
1,at://did:plc:b/app.bsky.feed.post/2,bo.bsky.social,2025-08-26 05:00:00+00:00,with a picture,10
2,at://did:plc:a/app.bsky.feed.post/3,ana.bsky.social,2025-08-27 09:15:00+00:00,third post,0


In [36]:
table.dtypes

uri                           str
handle                        str
created_at    datetime64[us, UTC]
text                          str
likes                       int64
dtype: object

Two rules for this conversion:

- Keep the raw JSON. The table is derived from it, and the next question will need a field you did not keep.
- Do not force everything into one table. Posts, authors, and embedded images are three tables joined by IDs. That is where the data management sessions start.

In [37]:
# Clean up the files this notebook created.
for name in ["quoted.csv", "sample_copy.csv", "sample_pipe.csv", "sample_tab.tsv", "fips.csv.gz",
             "fips.parquet", "fips.jsonl", "big.csv", "big.csv.gz", "big.parquet", "big_zstd.parquet"]:
    pathlib.Path("data", name).unlink(missing_ok=True)